# Outline
2022/05/01-2022/05/03

Master its specific usage, implementation methods, use cases, and time/space complexity

Hash tables: combine with example problems, summarize various clever applications, and what properties of hash tables they leverage

Key knowledge points:
- Hash Function
- Collision Resolution / load factor

What important real-world applications exist

Main references:
- https://www.jianshu.com/p/67b825e08d17
- https://zhuanlan.zhihu.com/p/63142005
- https://leetcode-cn.com/tag/hash-table/problemset/
- https://blog.csdn.net/Avery123123/article/details/103583115

`from collections import Counter`
`Counter()`

# Theory

- [X] What is a hash table?
- [X] How is it implemented?
- [X] Time and space complexity
- [X] Hash function
- [X] Hash collisions and resolution

A hash table is essentially an "array" that stores key-value pairs. It's very helpful for lookups!!! A data structure that trades space for time.
- Most direct form: for the value being stored, use a hash function to map its key to a memory cell. When storing, put the value directly in that memory cell.
- Variant (hashset): the value itself acts as the key; use a hash function to map its key to a memory cell. When storing, put 1 directly in that memory cell. When deleting, put 0 (or some other distinguishing value) in that memory cell.

There are three objects here:
- key: uniquely identifies the data object — can be the data object itself, or some subset of its attributes.
    - [X] How is the key chosen? For simple data types, it's usually the value itself; for complex ones, an id can be constructed manually. In Python, immutable objects are hashable — no matter how complex the object is, immutability means different objects have different addresses, so their addresses can be used to compute the hash.
- value: the complete data object being stored
- Memory cell: the memory cell stores the value. This isn't absolute either — for example, in Python's dict implementation, each key occupies a slot, and a slot is split into two parts: a reference to the key and a reference to the value. A hash function is used to get the key's hash value, which is then taken modulo the array length to get the storage index. The benefit of storing the key this way is that it makes rehashing (growing/shrinking) easier.

Further reading: [Python's hashable](https://www.cnblogs.com/zhouie/p/10702613.html)


Key points:
- [Hash function](https://zhuanlan.zhihu.com/p/63142005): its role is mapping a key to a memory cell. The actual implementation is often split into two steps: step one, the hash computation, maps the key to a hash value (this is where the real hash function operates); step two, the modulo operation, takes the hash value modulo the size of the hash table to get the storage position within that table.
    - [X] How to choose or design one?
    - [X] If the object isn't a basic type (like a string) but a complex data structure, how do you design a hash function?
    - [X] What properties should a (good) hash function have?
- Hash table: the underlying storage for key/value. It's a contiguous block of memory used to store the value objects. The key doesn't need to be stored — it's only used for computation.
    - [X] How is fast lookup of a key implemented? Why can it be done in O(1) time?
        [link](https://leetcode-cn.com/tag/hash-table/problemset/) uses a hash function to compute the key, giving a data structure that directly accesses the storage location in memory based on the key.


   

## Hash Table Implementation Methods
Suppose we need to store n data elements. Set up a contiguous storage area of length m (m ≥ n). For each data element's key Ki (0 <= i <= n-1), use a hash function hash(Ki) to map Ki to some address hash(Ki) in memory, and store the data element there. This is actually an array-based implementation.

- [X] So does this mean a hash table actually needs more memory than the number of elements actually stored? How much more?
    Yes. Generally, a hash table has a default size when initialized. Once the table is filled to a certain ratio, it grows in size and performs a rehash. How much larger it needs to be isn't fixed — it's determined by the specific implementation (e.g. the specified load factor).
- [X] Does this mean I need to size the memory cell based on the size of the object? What if different value objects have different sizes?
    No. What a hash table's memory cell actually stores isn't the value itself, but a pointer to the value, and pointers all have a uniform size.



Various forms of underlying implementation:
- Most basic: array — a contiguous block of memory; the hash value obtained from the key, modulo the hash table's size, gives the storage address. Downside: can't resolve hash collisions — reducing collisions requires a large amount of extra space.
- More advanced: array + array/linked list — an extension to resolve hash collisions: when two keys map to the same memory cell, extra storage is used to hold both simultaneously.
    - Array: can be thought of as multi-level hashing. The first hash value locates a sub-array; hashing continues within the sub-array to get the exact position, and so on — nesting can go multiple levels as needed. Downside: a large amount of extra space.
    - Linked list: the benefit is O(1) insertion; the downside is O(n) lookup.
- Advanced: array + linked list/red-black tree — when a particular bucket (the outermost inner cell) holds a lot of values, using a linked list makes lookup too slow, so a height-balanced binary tree can be used instead. The benefit is lookup complexity drops to O(logN), but insertion complexity also rises to O(logN).


Various applications
- HashMap: stores value using key. Underlying implementation of Python's dict. External interface:
    - add(key, val) adds a new key-val pair
    - delete(key) deletes that key and its corresponding value
    - find(key) checks whether the key exists
- HashSet: stores items and quickly determines whether an item is in the current set. Underlying implementation of Python's set. External interface:
    - add(item) adds a new item
    - delete(item) deletes that item
    - find(item) checks whether the item exists

In [2]:
# Version 1.0 HashMap

class HashMap(object):
    def __init__(self, init_size=100): # init_size: the initial array size
        self._size = init_size
        self._table = [None for _ in range(self._size)]
    
    def hash_func(self, key): # the hash function used here is the most basic: direct modulo
        assert key is int
        return key % self._size
        
    def add(self, key, val):
        loc = self.hash_func(key)
        self._table[loc] = val
        
    def delete(self, key):
        loc = self.hash_func(key)
        self._table[loc] = None
    
    def find(self, key):
        loc = self.hash_func(key)
        return self._table[loc] # if exists, return the value; otherwise, return None
        
        
# Version 1.0 HashSet

class HashSet(object):
    def __init__(self, init_size=100): # init_size: the initial array size
        self._size = init_size
        self._table = [0 for _ in range(self._size)]
    
    def hash_func(self, key): # the hash function used here is the most basic: direct modulo
        assert key is int
        return key % self._size
        
    def add(self, value):
        loc = self.hash_func(value)
        self._table[loc] = 1 # unlike a map, what's stored here isn't the value — it's 0 or 1, used to flag whether it exists.
        
    def delete(self, value):
        loc = self.hash_func(value)
        self._table[loc] = 0
    
    def find(self, value):
        loc = self.hash_func(value)
        return self._table[loc] == 1 # if exists, return True; otherwise, return False
        
        


In [3]:
# Version 2.0 HashMap, handling collisions

class HashMap(object):
    def __init__(self, init_size=100, sub_size=100):
        self._size = init_size
        self._sub_size = sub_size # use an array/linked list to store multiple values on collision. In a more advanced 3.0 version, a binary tree/red-black tree could be used here instead.
        self._table = [[None for _ in range(self._sub_size)] for _ in range(self._size)]
    
    def hash_func(self, key, size):
        assert key is int
        return key % _size
        
    def add(self, key, val):
        loc = self.hash_func(key, self._size)
        sub_loc = self.hash_func(key, self._sub_size)
        self._table[loc][sub_loc] = val
        
    def delete(self, key):
        loc = self.hash_func(key)
        sub_loc = self.hash_func(key, self._sub_size)
        self._table[loc][sub_loc] = None
    
    def find(self, key):
        loc = self.hash_func(key)
        sub_loc = self.hash_func(key, self._sub_size)
        return self._table[loc][sub_loc]
        
        
# Version 2.0 HashSet

class HashSet(object):
    def __init__(self, init_size=100, sub_size=100): 
        self._size = init_size
        self._sub_size = sub_size # use an array/linked list to store multiple values on collision
        self._table = [[0 for _ in range(self._sub_size)] for _ in range(self._size)]
    
    def hash_func(self, key, size):
        assert key is int
        return key % size
        
    def add(self, value):
        loc = self.hash_func(value)
        sub_loc = self.hash_func(key, self._sub_size)
        self._table[loc][sub_loc] = 1
        
    def delete(self, value):
        loc = self.hash_func(value)
        sub_loc = self.hash_func(key, self._sub_size)
        self._table[loc][sub_loc] = 0
    
    def find(self, value):
        loc = self.hash_func(value)
        sub_loc = self.hash_func(key, self._sub_size)
        return self._table[loc][sub_loc] == 1 
        


## Time and Space Complexity


Overall, a hash table's space complexity is O(n) — it grows linearly with the number of stored elements.

A hash table's time complexity depends heavily on its design.

Array + linked list/array design:
Most of us have probably already used an array within each bucket to store values that land in the same bucket,
- Ideally, when the bucket size is small enough, it can be treated as a constant. Both insertion and search have O(1) time complexity.
- In the worst case, the maximum bucket size will be N. Insertion is O(1), but search is O(N).

Array + linked list + binary tree/red-black tree design:

If a single bucket ends up with too many values, they're kept in a height-balanced binary search tree instead.
- The average time complexity for insertion and search remains O(1).
- In the worst case, insertion and search are O(logN)

[Reference on red-black tree complexity](https://www.jianshu.com/p/d5dd618014f0)

## Hash Function

- [X] What properties should a (good) hash function have?
    - Efficient: the computation should be as simple as possible, to improve insertion/retrieval efficiency for the hash table
    - Uniform distribution: to reduce the probability of hash collisions
    - Compressive: to save memory
    - Consistent: repeated computations produce the same result.
- [X] What hash functions are available to choose from? [reference](https://zhuanlan.zhihu.com/p/101390996)
    - MD5 (Message-Digest Algorithm 5)
    - SHA-1
    - SHA-2: SHA-224, SHA-256, SHA-384, and SHA-512 are collectively known as SHA-2. [SHA-256](https://blog.csdn.net/u011583927/article/details/80905740) takes any string of data as input and produces a 256-bit (32-byte) hash value, usually represented as a 64-character hexadecimal string.
    - And so on — see cryptography references for details! Won't go into it further here — will study it later when I have time!
- [X] How do you hash different types of keys? In Python, as long as an object is immutable, its address can be used to compute the hash.
- [X] If the object isn't a basic type (like a string) but a complex data structure, how do you design a hash function?
    A hash function actually operates on an address — that way, even for a complex data structure, its address always has a uniform format.





## Hash Collisions and Resolution Strategies

Hash collision: in a word, different keys get hashed to the same location.

Resolution strategies:
- Chaining: this is the 2.0 version of the implementation above — in short, if there's a collision, just store everything together. It's a test of memory usage.
- Open addressing: search the array in some way for **an empty slot to place the item**. Basic idea: when the key's hash address p = H(key) collides, use p as a base to generate another hash address p1; if p1 still collides, use p as a base again to generate another hash address p2, ..., until a non-colliding hash address pi is found.
    - [X] How does lookup work? The precondition for lookup to work is that the memory cell stores part of the key (e.g. in Python's dict implementation: the key's index is stored). Then, while walking along the sequence of indices, if a memory cell is already occupied, you can compare the stored key against the key being searched for to decide whether you've found it, and whether to continue. Lookup fails once an empty cell is reached.
    - [X] How does deletion work? [reference](https://www.cnblogs.com/east7/p/12594894.html) You can't truly delete — instead you mark it, otherwise it would cause lookup errors. The slot is only truly considered deleted once it's filled again on a subsequent insert.

# Key LeetCode Problems

Selected from the leetcode cookbook

- [003 Longest Substring Without Repeating Characters](https://leetcode-cn.com/problems/longest-substring-without-repeating-characters/). A classic sliding window — the condition the window maintains is that it contains no repeated characters internally.
    - Can move the left boundary to remove repeated characters from the window (time complexity 2n);
    - Or use a hash bucket to store the last seen position of each character, so that on a repeat, the window's next left boundary can be obtained directly (time complexity n).
- [030 Substring with Concatenation of All Words](https://leetcode-cn.com/problems/substring-with-concatenation-of-all-words/). Starred!!! Worth reviewing. The overall idea is a sliding window, treating words as characters for simplicity, aided by a hash table for counting. The difficulty: characters within a word may repeat consecutively, so if you slide in units of a whole word's length, you'll miss many scenarios. How to solve it? Add an outer loop that runs for the length of a single word. How the hash table is used here:
    - Reference solution: used as a hash counter — simple and easy to understand, for counting. But it has a redundant operation on deletion.
    - Index-tracking solution: stores the index corresponding to each word, optimizing away the redundant deletion operation, but requires checking valid indices every time.
- [076 Minimum Window Substring](https://leetcode-cn.com/problems/minimum-window-substring/). Also a sliding window. The key is clarifying the condition the window maintains, and how to move the window! The hash table's role is counting.
- [128 Minimum Window Substring](https://leetcode-cn.com/problems/longest-consecutive-sequence/). Starred! Couldn't solve it the first time. What the hash table stores: the length of the longest consecutive interval containing that point.
- [720 Longest Word in Dictionary](https://leetcode-cn.com/problems/longest-word-in-dictionary/). Very simple idea: check whether every prefix substring has appeared; there's a lot of room for implementation optimization. Relation to hash tables: can be used as a set to check existence. My implementation actually didn't use a hash table — it used an array for lookup instead.
- [726 Number of Atoms](https://leetcode-cn.com/problems/number-of-atoms/). Overall idea: a problem about bracket sequences: solved via recursion or a stack. Tree structure, expanded recursively.
- [930 Binary Subarrays With Sum](https://leetcode-cn.com/problems/binary-subarrays-with-sum/). Starred!!! I actually got stuck on this one. A clever application of prefix sums + taking the difference of two sliding windows. The property exploited: the prefix sum is non-decreasing, and elements are only 0 or 1. Consider a variant: what if elements can be positive integers or 0, not just 0/1 — how would you solve that? You'd need this check `if s1 == goal:`, which can be omitted in the current 0/1 setting.
- [992 Subarrays with K Different Integers](https://leetcode-cn.com/problems/subarrays-with-k-different-integers/). Starred!!! Essentially a sliding window, same as 930, maintaining two sliding windows. One sliding window guarantees the longest subarray with k distinct integers, the other guarantees the longest subarray with k-1 distinct integers; the difference between the two indices gives the range of valid left endpoints for the subarray. To reduce complexity: move the right side in a single loop, and move the left side based on the window-maintenance condition. For sliding window problems, the key is understanding the condition the window maintains. The hash table's role here: counting.


In [ ]:
# 030 
# Reference solution
class Solution:
    def findSubstring(self, s: str, words: List[str]) -> List[int]:
        from collections import Counter
        if not s or not words:return []
        one_word = len(words[0])
        word_num = len(words)
        n = len(s)
        if n < one_word:return []
        words = Counter(words)
        res = []
        for i in range(0, one_word): # outer loop: runs for the length of a single word
            cur_cnt = 0
            left = i
            right = i
            cur_Counter = Counter()
            while right + one_word <= n:
                w = s[right:right + one_word]
                right += one_word
                if w not in words: # when the current word isn't in the given list, clear and start over
                    left = right
                    cur_Counter.clear()
                    cur_cnt = 0
                else:
                    cur_Counter[w] += 1
                    cur_cnt += 1
                    while cur_Counter[w] > words[w]: # if a word's count exceeds the given count, remove earlier occurrences. There's a redundant operation here.
                        left_w = s[left:left+one_word]
                        left += one_word
                        cur_Counter[left_w] -= 1
                        cur_cnt -= 1
                    if cur_cnt == word_num :
                        res.append(left)
        return res


In [ ]:
# Index-tracking solution
from collections import Counter
class Solution:
    def findSubstring(self, s: str, words: List[str]) -> List[int]:
        words_num = Counter(words)
        start_idx = []
        unit_size = len(words[0])
        for start_i in range(unit_size):
            start_id = start_i # tracks the currently valid starting position
            words_loc = {} # records the positions where each word appears
            for i in range(start_i, len(s), unit_size):
                cur_word = s[i: (i+unit_size)] 
                if cur_word not in words_num:
                    start_id = i + unit_size
                    words_loc = {}
                    continue
                if cur_word in words_loc:
                    new_loc = []
                    for valid_id, saved_loc in enumerate(words_loc[cur_word]): # only keep records after the current starting position
                        if saved_loc >= start_id:
                            new_loc = words_loc[cur_word][valid_id: ]
                            break
                    new_loc.append(i)
                    words_loc[cur_word] = new_loc
                    if len(new_loc) > words_num[cur_word]:
                        start_id = words_loc[cur_word][0] + unit_size
                        words_loc[cur_word] = words_loc[cur_word][1:]
                else:
                    words_loc[cur_word] = [i]

                # judge if the current words_loc satisfies the condition
                if i - start_id == unit_size * (len(words)-1):
                    start_idx.append(start_id)
                    start_id += unit_size
        return start_idx

In [ ]:
# 076 My solution
"""
Approach:
- Solve with a sliding window, keeping time complexity at O(n)
- The window maintenance condition is: the number of t's characters contained in the window < the length of t. Once they're equal, first shrink the window to its "minimum", then check and save the result
- The tricky part here: one num tracks counts that don't exceed each character's count in t, while a counter tracks each character's actual occurrence count, used to shrink the window.
- Condition for shrinking the window to its "minimum": moving even one more step would break the condition.

"""


from collections import Counter
class Solution:
    def minWindow(self, s: str, t: str) -> str:
        if len(t) > len(s):
            return ""
        min_str = ""
        min_len = 0
        start_id = 0
        existed_num = 0
        ttl_num = len(t)
        t_cnt = Counter([c for c in t])
        cur_cnt = Counter()
        for end_id, end_char in enumerate(s):  # solve with a sliding window
            if end_char not in t:
                continue
            if cur_cnt[end_char] < t_cnt[end_char]:
                existed_num += 1
            cur_cnt[end_char] += 1
            if existed_num == ttl_num:
                while existed_num == ttl_num: # move the window, shrinking from the left, reducing the string length
                    if s[start_id] in t:
                        cur_cnt[s[start_id]] -= 1
                        if cur_cnt[s[start_id]] < t_cnt[s[start_id]]:
                            existed_num -= 1
                    start_id += 1
                if min_len == 0 or (end_id - start_id + 2) < min_len: # the actual start position is start_id - 1; start_id is actually the index one past the point where the window condition was first broken
                    min_len = end_id - start_id + 2
                    min_str = s[(start_id - 1): (end_id + 1)]
        return min_str

In [ ]:
# 128
class Solution:
    def longestConsecutive(self, nums: List[int]) -> int:
        max_len = 0
        max_lengths = dict()
        for num in nums:
            if num in max_lengths:
                continue
            left = max_lengths.get(num - 1, 0)
            right = max_lengths.get(num + 1, 0)
            cur_len = left + 1 + right
            max_len = max(max_len, cur_len)
            # print("Before update: ", max_lengths)
            max_lengths[num - left] = cur_len
            max_lengths[num + right] = cur_len
            max_lengths[num] = cur_len
            # print("After update: ",max_lengths)
        return max_len

In [ ]:
# 720
class Solution:
    def order_word(self, words):
        max_len = max([len(w) for w in words])
        len2words = [[] for _ in range(max_len + 1)]
        len2words[0].append('')
        for word in words:
            len2words[len(word)].append(word)
        return max_len, len2words
    def check_words(self, max_len, len2words):
        valid_words = [['']]
        for l in range(1, max_len+1):
            cur_words = []
            for word in len2words[l]:
                if word[:-1] in valid_words[-1]:
                    cur_words.append(word)
            if len(cur_words) == 0:
                return valid_words
            valid_words.append(sorted(cur_words))
        return valid_words
        


    def longestWord(self, words: List[str]) -> str:
        max_len, len2words = self.order_word(words) 
        valid_words = self.check_words(max_len, len2words)
        return valid_words[-1][0]


In [ ]:
# 726

from collections import Counter
class Solution:
    def split_formula(self, formula): # return a list of items like (num, str), num: the frequence of the str
        if "(" not in formula:
            return [[1, formula]]
        splited = []
        left_cnt = 0
        right_cnt = 0
        within = True
        num_str = ""
        start_id = 0
        for end_id, cur_char in enumerate(formula):
            if cur_char == "(":
                if left_cnt == 0 and start_id != end_id:
                    num = 1 if num_str == "" else int(num_str)
                    if formula[start_id] == "(":
                        splited.append([num, formula[start_id + 1: end_id - len(num_str) - 1]])
                    else:
                        splited.append([num, formula[start_id: end_id - len(num_str)]])
                    start_id = end_id
                    num_str = ""
                    within = True
                left_cnt += 1
            elif cur_char == ")":
                left_cnt -= 1
                if left_cnt == 0:
                    within = False
            elif cur_char.isdigit() and not within:
                num_str += cur_char
            else:
                if cur_char.isupper() and (num_str != "" or formula[end_id - 1] == ")"):
                    num = 1 if num_str == "" else int(num_str)
                    if formula[start_id] == "(":
                        splited.append([num, formula[start_id + 1: end_id - len(num_str) - 1]])
                    else:
                        splited.append([num, formula[start_id: end_id - len(num_str)]])
                    start_id = end_id
                    num_str = ""
                    within = True

                continue
        num = 1 if num_str == "" else int(num_str)
        if formula[start_id] == "(":
            splited.append([num, formula[start_id + 1: end_id - len(num_str)]])
        else:
            splited.append([num, formula[start_id: end_id - len(num_str) + 1]])
        return splited

    def isBasicFormula(self, formula):
        return "(" not in formula

    def splitbyBigAlpha(self, formula):
        alpha_list = []
        start_id = 0
        for i in range(1, len(formula)):
            if formula[i].isupper():
                alpha_list.append(formula[start_id: i])
                start_id = i
        alpha_list.append(formula[start_id: ])
        return alpha_list
    
    def extract_number(self, alpha):
        pure_alpha = alpha
        num = 1
        for i in range(len(alpha)):
            if alpha[i].isdigit():
                pure_alpha = alpha[:i]
                num = int(alpha[i:])
                break
        return num, pure_alpha

    def calcBasic(self, formula):
        alpha_list = self.splitbyBigAlpha(formula)
        counter = Counter()
        # print(alpha_list)
        for alpha in alpha_list:
            num, pure_alpha = self.extract_number(alpha)
            counter[pure_alpha] += num
        return counter

    def dictCount(self, formula):
        count = Counter()
        formula_list = self.split_formula(formula)
        for form in formula_list:
            if self.isBasicFormula(form[1]):
                cur_count = self.calcBasic(form[1])
            else:
                cur_count = self.dictCount(form[1])
            for key, val in cur_count.items():
                cur_count[key] = val * form[0]
            count.update(cur_count)
        return count

    def Counter2Str(self, counter):
        res = ""
        keys = sorted(counter.keys())
        for key in keys:
            val = counter[key]
            if val == 1:
                res += key
            else:
                res += (key + str(val))
        return res

    def countOfAtoms(self, formula: str) -> str:
        counter = self.dictCount(formula)
        return self.Counter2Str(counter)

In [ ]:
# 930
class Solution:
    def numSubarraysWithSum(self, nums: List[int], goal: int) -> int:
        cnt = 0
        l1 = 0
        l2 = 0
        s1 = 0
        s2 = 0
        for r in range(len(nums)):
            s1 += nums[r]
            s2 += nums[r]
            while s1 > goal and l1 <= r:
                s1 -= nums[l1]
                l1 += 1
            while s2 >= goal and l2 <= r:
                s2 -= nums[l2]
                l2 += 1
            if s1 == goal: # this check can be omitted.
                cnt += (l2 - l1)
        return cnt


In [ ]:
# 992
class Solution:
    def subarraysWithKDistinct(self, nums: List[int], k: int) -> int:
        res = 0
        l1 = 0
        l2 = 0 # the ideal window's left side is within [l1, l2), [l1, r]
        l1_diff = 0
        l2_diff = 0
        l1_count = dict()
        l2_count = dict()
        for r, num in enumerate(nums):
            l1_count[num] = l1_count.get(num, 0) + 1
            l2_count[num] = l2_count.get(num, 0) + 1
            if l1_count[num] == 1:
                l1_diff += 1
            if l2_count[num] == 1:
                l2_diff += 1
            while l2_diff == k:
                l2_count[nums[l2]] -= 1
                if l2_count[nums[l2]] == 0:
                    l2_diff -= 1
                l2 += 1
            while l1_diff == k + 1:
                l1_count[nums[l1]] -= 1
                if l1_count[nums[l1]] == 0:
                    l1_diff -= 1
                l1 += 1
            if l1_diff == k: # without the last item, the diff num is k.
                res += l2 - l1
        return res


# Real-World Applications

- Hash tables are the underlying implementation of Python's dict and set structures
- Some ID generation schemes use the murmurhash function (leverages uniformity, is reversible)
- Some KV database implementations: how does Redis store data? http://redisbook.com/preview/dict/rehashing.html
- Application in Bitcoin: the hash function used is SHA-256 (Secure Hash Algorithm). (irreversible)
- Feature values generated from hashing: LSH (Locality Sensitive Hashing): design a hash function that computes hash values for two points such that they have a high probability of being equal when the points are close, and if the two points are far apart, the probability of them sharing the same hash value is low. (non-uniform, keeps similar things close)
    [Idea](https://www.jianshu.com/p/d4368c8f40cb): if we can roughly bucket users first, so that potentially similar users have a high probability of landing in the same bucket, then each user's "candidate set of similar users" becomes relatively small, reducing the computational cost of finding similar users. LSH is exactly this kind of approximate algorithm. The core idea is **hash bucketing**.
- Bloom filter: uses multiple independent hash functions for fast lookups. Has false positives, following the principle of "better to over-flag than to miss."

# My Summary

In LeetCode problems, hash tables are commonly used for:
- Counting, using `from collections import Counter`; common statements: `cur_cnt = Counter()# for initialization `, `cur_cnt.clear()`, `cur_cnt[key] += 1` (the key doesn't need to already exist — it defaults to starting from 0)
- Recording, using other default-dict types: `from collections import defaultdict`, e.g. defaulting to an empty list: `dd = defaultdict(list)`